In [1]:
import numpy as np
import nbimporter
import cepairsimplementation as ce
from reci import RECI
from lingam import lingam
from igci import IGCI
from anm import anm
from pnl import pnl
from cgnn import cgnn
from emd import emd
import os
import glob
import matplotlib.pyplot as plt
import traceback
import functools
from scipy.optimize import nnls
import pandas as pd
from sklearn.linear_model import Lasso, Ridge
import json
import math
import csv
import synthetic_nn_keras as synth_k
import re
import cvxpy as cp
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [ ]:
def getSynthetic(dir):
    data_list = []
    txt_files = glob.glob(os.path.join(dir, '*.txt'))
    
    for file_path in txt_files:
        # Load numerical data from text file into a numpy array
        data = np.loadtxt(file_path)
        
        # Verify there are exactly 2 columns
        if data.ndim != 2 or data.shape[1] != 2:
            continue  # Skip files with invalid format
        
        # Extract columns and convert to Python lists
        x = np.array(data[:, 0]).reshape(-1, 1)
        y = np.array(data[:, 1]).reshape(-1, 1)
        
        data_list.append([x,y])
    return data_list


def getOld(dataset):
    folder="../other implementations/synthetic_datasets"
    pairs_file   = f"{folder}/{dataset}_pairs.csv"
    targets_file = f"{folder}/{dataset}_targets.csv"
    
    # --- sanity checks ---
    if not os.path.isfile(pairs_file):
        raise FileNotFoundError(f"Pairs file not found: {pairs_file}")
    if not os.path.isfile(targets_file):
        raise FileNotFoundError(f"Targets file not found: {targets_file}")
    
    # --- read in with pandas ---
    df_pairs   = pd.read_csv(pairs_file)
    df_targets = pd.read_csv(targets_file)
    
    if len(df_pairs) != len(df_targets):
        raise ValueError(
            f"Row count mismatch: {len(df_pairs)} in pairs vs "
            f"{len(df_targets)} in targets"
        )
    
    data_list = []
    for idx, pair_row in df_pairs.iterrows():
        # get the raw space‐separated strings
        x_str = str(pair_row.iloc[1])
        y_str = str(pair_row.iloc[2])
        
        # split & convert to floats, then make column vectors
        x = np.array([float(v) for v in x_str.split()]).reshape(-1, 1)
        y = np.array([float(v) for v in y_str.split()]).reshape(-1, 1)
        
        # swap if target's 2nd column is -1
        if df_targets.iloc[idx, 1] == -1:
            x, y = y, x
        data_list.append([x, y])
    
    return data_list


In [3]:
def run_synthetic(func,dataset, **kwargs):
    if dataset[:2] == "CE":
        data = getOld(dataset)
    else:
        dir=f"generate_txt/{dataset}"
        data = getSynthetic(dir)
    for d in data:
        if np.isnan(d).any():
            print("Found NaN in data:", dataset, d)
    weights=np.ones(len(data))/len(data)
    scores =np.array([func(d,**kwargs) for d in data])
    kwargs_str = ", ".join(f"{k}={v}" for k, v in kwargs.items())
    folder=f"predictions/{func.__name__}_{kwargs_str}"
    os.makedirs(folder, exist_ok=True)
    np.savetxt(f"{folder}/{dataset}.txt", np.column_stack((scores, weights)))

def analyse_predictions(funcname,dataset, **kwargs):
    kwargs_str = ", ".join(f"{k}={v}" for k, v in kwargs.items())
    folder=f"predictions/{funcname}_{kwargs_str}"
    data = np.loadtxt(f"{folder}/{dataset}.txt")
    # Split the data into scores and weights
    scores = data[:, 0]  # First column
    weights = data[:, 1]  # Second column
    
    #AUROC
    y_scores, y_true = ce.switch_signs(scores)
    normalized_y=ce.minmax_scale(y_scores)
    normalized_y[np.isnan(normalized_y)] = 0.5
    auroc = ce.roc_auc_score(y_true,normalized_y,sample_weight=weights)
    #Accuracy
    guess=ce.sign_to_binary(scores)<0
    accuracy = sum(guess*weights)/sum(weights)


    #plot
    ce.save_excel(funcname,dataset,[auroc,accuracy],["Auroc","Accuracy"])


In [ ]:
def analyse_exchangeable_dataset(method):
    acc_weights, labels=synth_k.load_weights(filename='data/dataset_weights.csv', metric='Accuracy')
    aur_weights, labels=synth_k.load_weights(filename='data/dataset_weights.csv', metric='Auroc')
    method=method+"_"
    full_scores=[]
    aur_finalw=[]
    acc_finalw=[]
    in_auroc=[]
    in_auroc_weights=[]
    if method.startswith('funcR_'):
        return np.nan
    for aur_w, acc_w,label in zip(aur_weights, acc_weights, labels):
        file_path = f"predictions/{method}/{label}.txt"
        if not os.path.exists(file_path):
            continue
        data = np.loadtxt(file_path)
        # Split the data into scores and weights
        scores = data[:, 0]  # First column
        full_scores.extend(scores)
        weights = data[:, 1]  # Second column
        #print(weights)
        aur_finalw.extend(aur_w*weights)
        acc_finalw.extend(acc_w*weights)
        #AUROC-in
        y_scores, y_true = ce.switch_signs(scores)
        normalized_y=ce.minmax_scale(y_scores)
        normalized_y[np.isnan(normalized_y)] = 0.5
        auroc = ce.roc_auc_score(y_true,normalized_y,sample_weight=weights)
        in_auroc.append(auroc)
        in_auroc_weights.append(aur_w)
    full_scores = np.array(full_scores)
    aur_finalw = np.array(aur_finalw)
    acc_finalw = np.array(acc_finalw)
    #in-AUROC
    in_auroc = np.array(in_auroc)
    in_auroc_weights = np.array(in_auroc_weights)
    in_auroc = sum(in_auroc * in_auroc_weights) / sum(in_auroc_weights)
    #AUROC
    y_scores, y_true = ce.switch_signs(full_scores)
    normalized_y=ce.minmax_scale(y_scores)
    normalized_y[np.isnan(normalized_y)] = 0.5
    auroc = ce.roc_auc_score(y_true,normalized_y,sample_weight=aur_finalw)
    #Accuracy
    guess=ce.sign_to_binary(full_scores)
    accuracy = sum(guess*acc_finalw)/sum(acc_finalw)
    return auroc, accuracy

def test_exchangeable_synthetic(method):
    _, labels=synth_k.load_weights(filename='data/dataset_weights.csv', metric='Accuracy')
    for dataset in labels:
        run_synthetic(method,dataset)
        analyse_predictions(method.__name__,dataset)
    return analyse_exchangeable_dataset(method.__name__)
    